# ReAct Agent
By: Thandokuhle Brian Msane

First, ReAct stands for reasoning and acting. So, this is an agent uses a loop attached to any number of tools. The LLM, which can be thought of as the brain of the Agent decides which tool to select and also go the end part when there is not tool left to call.

The objectives are
- learn how to build tools in LangGraph
- create a ReAct graph
- work with differnt types of messages such as `ToolMessage`, `SystemMessges`, and `BaseMessages`
- create and test a robust agent

In [ ]:
from typing import Annotated, Sequence, TypedDict

from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages # reducer function that allows us to append everything into the state
from langgraph.prebuilt import ToolNode

load_dotenv()

True

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    

@tool
def add(a: int, b: int) -> int:
    """Addition function that adds together two numbers"""
    return a + b

@tool
def subtract(a: int, b: int) -> int:
    """Subtract two numbers"""
    return a - b

@tool
def mul(a: int, b: int) -> int:
    """multiply two numbers"""
    return a * b

tools = [add, subtract, mul]
model = ChatOpenAI(model='gpt-4o-mini').bind_tools(tools)


def model_call(state: AgentState) -> AgentState:
    """Answer the query using an LLM call"""
    system_message = SystemMessage("you are an AI assistant, please answer my query to the best of your ability")
    response = model.invoke([system_message] + state['messages'])
    return {'messages': [response]}


def should_continue(state: AgentState):
    """Decide if we should go for a tool call or just end"""
    last_message = state['messages'][-1]
    if not last_message.tool_calls:
        return 'end'
    return 'continue'

    
graph = StateGraph(AgentState)
graph.add_node('our_agent', model_call)
graph.add_node('tool_node', ToolNode(tools=tools))

graph.add_conditional_edges(
    'our_agent',
    should_continue,
    path_map={
        'end': END,
        "continue": 'tool_node'
    }
)
graph.add_edge('tool_node', 'our_agent')
graph.add_edge(START, 'our_agent')
app = graph.compile()

def print_stream(stream):
    for s in stream:
        message = s['messages'][-1]
        if isinstance(message, type):
            print(message)
        else:
            message.pretty_print()

inputs = {
    'messages': [('user', 'Add 4 and 7 and then multiply the result by 6')]
}
print_stream(app.stream(inputs, stream_mode='values'))